In [5]:
# import datashader as ds
import plotly.express as px
import polars as pl

pl.Config.set_tbl_width_chars(256)
pl.Config.set_fmt_str_lengths(256)
pl.Config.set_tbl_rows(24)

# Location of the IPC generated files from above.
ipc_dir = "/home/seb/git/s2protocol-rs/ipcs"

unit_born_df = pl.scan_ipc(f"{ipc_dir}/unit_born.ipc")
unit_died_df = pl.scan_ipc(f"{ipc_dir}/unit_died.ipc")
stats_df = pl.scan_ipc(f"{ipc_dir}/stats.ipc")
upgrades_df = pl.scan_ipc(f"{ipc_dir}/upgrades.ipc")
lobby_slot_init_data_df = pl.scan_ipc(f"{ipc_dir}/lobby_init_data.ipc")
user_init_data_df = pl.scan_ipc(f"{ipc_dir}/user_init_data.ipc")
details_df = pl.scan_ipc(f"{ipc_dir}/details.ipc")
cmd_target_point_df = pl.scan_ipc(f"{ipc_dir}/cmd_target_point.ipc")
cmd_target_unit_df = pl.scan_ipc(f"{ipc_dir}/cmd_target_unit.ipc")

player_1 = "Serral"
player_2 = "Classic"
test_player = "Sazed"


def ext_fs_id_by_player_name(player_name):
    return details_df.filter(
        (pl.col("player_name").eq(player_name) & pl.col("player_observe").eq(0))
    ).select("ext_fs_id")

player1_games = ext_fs_id_by_player_name(player_1)
player2_games = ext_fs_id_by_player_name(player_2)

details_df.join(player1_games, on="ext_fs_id").join(player2_games, on="ext_fs_id").sort("ext_datetime", descending=True).head(10).collect()


player_name,player_toon_region,player_toon_program_id,player_toon_realm,player_toon_id,player_race,player_color_a,player_color_r,player_color_g,player_color_b,player_control,player_team_id,player_observe,player_result,player_working_set_slot_id,player_hero,title,is_blizzard_map,time_utc,time_local_offset,ext_fs_id,ext_datetime
str,u8,u32,u32,u64,str,u8,u8,u8,u8,u8,u8,u8,str,u8,str,str,bool,i64,i64,u64,datetime[ns]
"""Classic""",2,21298,1,9691041,"""Protoss""",255,0,66,255,2,1,0,"""Loss""",14,"""""","""Magannatha LE""",true,133979400586194635,108000000000,38040,2025-07-25 15:00:58.619463
"""Serral""",2,21298,1,9691277,"""Zerg""",255,180,20,30,2,0,0,"""Win""",13,"""""","""Magannatha LE""",true,133979400586194635,108000000000,16737,2025-07-25 15:00:58.619463
"""Classic""",2,21298,1,9691041,"""Protoss""",255,0,66,255,2,1,0,"""Loss""",14,"""""","""Magannatha LE""",true,133979400586194635,108000000000,16737,2025-07-25 15:00:58.619463
"""Serral""",2,21298,1,9691277,"""Zerg""",255,180,20,30,2,0,0,"""Win""",13,"""""","""Magannatha LE""",true,133979400586194635,108000000000,38040,2025-07-25 15:00:58.619463
"""Classic""",2,21298,1,9691041,"""Protoss""",255,0,66,255,2,1,0,"""Loss""",11,"""""","""Torches LE""",true,133979387828298866,108000000000,16736,2025-07-25 14:39:42.829886
"""Serral""",2,21298,1,9691277,"""Zerg""",255,180,20,30,2,0,0,"""Win""",12,"""""","""Torches LE""",true,133979387828298866,108000000000,16736,2025-07-25 14:39:42.829886
"""Classic""",2,21298,1,9691041,"""Protoss""",255,0,66,255,2,1,0,"""Loss""",11,"""""","""Torches LE""",true,133979387828298866,108000000000,21005,2025-07-25 14:39:42.829886
"""Serral""",2,21298,1,9691277,"""Zerg""",255,180,20,30,2,0,0,"""Win""",12,"""""","""Torches LE""",true,133979387828298866,108000000000,21005,2025-07-25 14:39:42.829886
"""Classic""",2,21298,1,9691041,"""Protoss""",255,0,66,255,2,1,0,"""Win""",13,"""""","""Ley Lines""",true,133979380275768376,108000000000,16739,2025-07-25 14:27:07.576837


In [11]:
# Magannatha EWC grand final (last game) is ext_fs_id 16736
chosen_fs_id = 31276  # One of the above
details_lobby_df = details_df.filter([pl.col("ext_fs_id").eq(chosen_fs_id)]).join(
    lobby_slot_init_data_df.filter([pl.col("ext_fs_id").eq(chosen_fs_id)]).with_columns(
        pl.col("working_set_slot_id").alias("player_working_set_slot_id")
    ),
    on="player_working_set_slot_id",
)
details_lobby_df.collect_schema()

Schema([('player_name', String),
        ('player_toon_region', UInt8),
        ('player_toon_program_id', UInt32),
        ('player_toon_realm', UInt32),
        ('player_toon_id', UInt64),
        ('player_race', String),
        ('player_color_a', UInt8),
        ('player_color_r', UInt8),
        ('player_color_g', UInt8),
        ('player_color_b', UInt8),
        ('player_control', UInt8),
        ('player_team_id', UInt8),
        ('player_observe', UInt8),
        ('player_result', String),
        ('player_working_set_slot_id', UInt8),
        ('player_hero', String),
        ('title', String),
        ('is_blizzard_map', Boolean),
        ('time_utc', Int64),
        ('time_local_offset', Int64),
        ('ext_fs_id', UInt64),
        ('ext_datetime', Datetime(time_unit='ns', time_zone=None)),
        ('ext_fs_id_right', UInt64),
        ('ext_fs_sha256', String),
        ('ext_fs_file_name', String),
        ('control', Int64),
        ('user_id', Int64),
        ('team_id',

In [14]:
# player1_build_order = (
#     cmd_target_point_df
#         .filter(
#             pl.col("ext_fs_id").eq(chosen_fs_id)
#         )
#         .join(
#             details_lobby_df.filter(
#                 pl.col("ext_fs_id").eq(chosen_fs_id)
#                 & pl.col("player_name").eq(player_1)
#                 & pl.col("player_observe").eq(0)
#             ),
#             on=["user_id", "ext_fs_id"],
#         )
# )

player1_build_order = (
    cmd_target_point_df
        .filter(
            pl.col("ext_fs_id").eq(chosen_fs_id)
        )
        .join(
            details_lobby_df.filter(
                pl.col("ext_fs_id").eq(chosen_fs_id)
                & pl.col("player_name").eq(player_1)
                & pl.col("player_observe").eq(0)
            ),
            on=["user_id", "ext_fs_id"],
        )
        .filter(
                (
                    pl.col("ability").str.starts_with("Build.") 
                    & ~pl.col("ability").str.starts_with("Build.Creep") # Remove Creep Tumors
                    & ~pl.col("ability").str.starts_with("Build.Spore") # Remove SporeCrawlers
                    & ~pl.col("ability").str.starts_with("Build.Spore") # Remove SpineCrawlers
                )
                | pl.col("ability").str.starts_with("Train.")
                | pl.col("ability").str.contains("spawn")
        )
        .sort(by=[pl.col("ext_replay_loop")])
        #.filter(pl.col("ability").eq("Build.LurkerDenMP")) # BUG: We have duplicates... checking...
).collect()
aoeu = (
    cmd_target_point_df
        .filter(
            pl.col("ext_fs_id").eq(chosen_fs_id)
            & pl.col("ext_replay_loop").eq(17783)
        )
)
aoeu.collect()

user_id,cmd_flags,abil_link,abil_cmd_index,ability,target_point_x,target_point_y,target_point_z,sequence,unit_group,unit_name,ext_replay_loop,ext_replay_seconds,ext_fs_id
i64,i64,u16,i64,str,i64,i64,i32,i64,u32,str,i64,u32,u64


In [13]:
lobby_slot_init_data_df.filter(pl.col("ext_fs_id").eq(chosen_fs_id) & pl.col("observe").eq(0)).collect()

ext_fs_id,ext_fs_sha256,ext_fs_file_name,control,user_id,team_id,observe,working_set_slot_id,map_size_x,map_size_y
u64,str,str,i64,i64,i64,u8,u8,u8,u8
31276,"""940f009c1bd2c1592b7698b11907a6e9b316c0d4dae3b4310e2f291d7a05469a""","""/home/seb/SCReplaysOnNVMe/SpawningTool/49/Classic v Serral: Game 5 - Persephone LE.SC2Replay""",2,1,0,0,1,160,184
31276,"""940f009c1bd2c1592b7698b11907a6e9b316c0d4dae3b4310e2f291d7a05469a""","""/home/seb/SCReplaysOnNVMe/SpawningTool/49/Classic v Serral: Game 5 - Persephone LE.SC2Replay""",2,3,1,0,12,160,184
